Build decompartmentalized model. Use decomp_compartments in options.yml to specify which compartments to decompartmentalize (e.g., {'m':'c'} to decompartmentalize mitochondrial proteins into cytosol).

Back up your input folder's files, since this script will overwrite them where needed.

# Make decompartmentalized GSM model

In [ ]:
# Consider reviewing spontaneous rxns identified by this process and adding them back into the original model, as these may occur in vivo in their newly added compartment
import cobra,json,os,sys,re,pandas as pd, io, requests
from bs4 import BeautifulSoup
from collections import OrderedDict
from copy import deepcopy

# keep going up until the "pycore" folder is found
wd = os.getcwd()
while not os.path.exists('pycore'):
	os.chdir('..')
base_path = os.getcwd()
sys.path.append(base_path)
from pycore import *

os.chdir(wd)

/Users/ejm6426/Documents/pkRBA/pycore/../build_model/input/iIsor850_RBA.json


In [2]:
decompartmentalize = True
decomp_spont_rxns = True # whether to decompartmentalize spontaneous reactions (i.e., those with GPRs in options.yml 'spont_GPRs' list)
# GPRs that prevent decompartmentalization of a reaction. Best to use all ambiguous ones, to avoid inaccurate comparisons between the original and decompartmentalized models. 
## example: rxns known to be spontaneous shouldn't need to be decompartmentalized since they can likely occur in any compartment where their metabolites appear (and would give an unfair disadvantage to the original model if the decompartmentalized model has it in more compartments), but ones that are non-spontanous may be worth decompartmentalizing.
GPRs_excluded = ['UNKNOWN', ''] 

rxns_added = []

# revert automated model reduction changes, to account for how decompartmentalization may have affected the original model's reactions (e.g., rxn A can be used only after decompartmentalization, but blocking in the original model would prevent its usage in the decompartmentalized model)
for r in model.reactions:
	for note in ('turned_off_for_RBA','altered_for_RBA'):
		if getattr(r, 'notes', {}).get(note, False):
			# set lower and upper bounds to "Original bounds were (x,y)"
			original_bounds = re.search(r'Original bounds were \(([^,]*),([^)]*)\)', r.notes.get(note,''))
			if original_bounds:
				r.lower_bound = float(original_bounds.group(1))
				r.upper_bound = float(original_bounds.group(2))
			# remove the note
			del r.notes[note]

# make decompartmentalized model for testing purposes
for k,v in RBA_defaults.decomp_compartments.items():
	print(f"Decompartmentalizing from {k} to {v}...")
	# for each rxn, if its compartments include the compartment to decompartmentalize from, add a new rxn with the same properties but with the new compartment instead of the old one (update any metabolites in the rxn accordingly); if the rxn is spontaneous and decomp_spont_rxns is True, add a new rxn with the same properties but with GPR 'SPONT' and the new compartment instead of the old one (update any metabolites in the rxn accordingly)
	for rxn in model.reactions:
		if rxn in model.boundary:
			continue
		if rxn.compartments == {k}:
			new_rxn_id = re.sub(f'_{k}$', f'_{v}', rxn.id)
			if new_rxn_id in model.reactions:
				continue
			new_rxn = deepcopy(rxn)
			new_rxn.id = new_rxn_id
			for m in new_rxn.metabolites:
				new_rxn.add_metabolites({m: -new_rxn.metabolites[m]})
			if not decomp_spont_rxns and rxn.gene_reaction_rule in RBA_defaults.spont_GPRs:
				continue
			if rxn.gene_reaction_rule in GPRs_excluded: # skip these since they're not technically spontaneous and we don't know their GPRs, so we can't be sure how to decompartmentalize them
				continue
			# add versions of each metabolite in the new compartment and update the rxn to use these new metabolites
			for met in rxn.metabolites:
				if met.compartment == k:
					new_met_id = re.sub(f'_{k}$', f'_{v}', met.id)
					if new_met_id in model.metabolites:
						new_met = model.metabolites.get_by_id(new_met_id)
					else:
						new_met = deepcopy(met)
						new_met.id = new_met_id
						new_met.compartment = v
						model.add_metabolites([new_met])
					new_rxn.add_metabolites({new_met: rxn.metabolites[met]})
			# check mass balance
			imbal = check_mass_balance_cobra(new_rxn, model)
			if any(imbal.values()):
				print(f"Warning: excluding new rxn {new_rxn.id} due to mass imbalance after decompartmentalization (imbalances: {imbal})")
				continue
			model.add_reactions([new_rxn])
			rxns_added.append(new_rxn_id)
			if rxn.gene_reaction_rule in RBA_defaults.spont_GPRs:
				print(f"Consider adding spontaneous rxn {new_rxn.id} to original model since it may occur in vivo in its newly added compartment")
# save decompartmentalized model for testing purposes
SM_path_decomp = os.path.join(wd, 'input', 'model_decompartmentalized.json')
cobra.io.save_json_model(model, SM_path_decomp)

Decompartmentalizing from m to c...
Decompartmentalizing from mm to c...


In [3]:
run_fva = False
if run_fva:
	fva = cobra.flux_analysis.flux_variability_analysis(model, rxns_added)
	# print any non-zero fluxes in the added reactions
	for rxn_id in rxns_added:
		if fva.loc[rxn_id, 'minimum'] != 0 or fva.loc[rxn_id, 'maximum'] != 0:
			print(f"Non-zero flux in added rxn {rxn_id}: min {fva.loc[rxn_id, 'minimum']}, max {fva.loc[rxn_id, 'maximum']}")

# RBA model updates

In [ ]:
# best to run this step the 1st time you decompartmentalize, to identify new PTM combinations the proteins could use. Not required for future runs, but still potentially useful in case databases are updated with new PTM information.
%run ./01_extract_enzymes_info_uniprot.ipynb

xt_c
nad_n and nad_r are the same
h_mm and h_g are the same
amet_e and amet_n are the same
2mbald_m and 3mbald_e are the same
dhap_rm and dhap_m are the same
fecost_e and fecost_c are the same
2mbac_e and iamac_c are the same
pi_rm and pi_m are the same
h2o_l and h2o_g are the same
quln_m and quln_c are the same
pi_e and pi_m are the same
g1p_c and f6p_c are the same
co2_n and co2_e are the same
fru_c and glc__D_e are the same
ile__L_e and ile__L_m are the same
hxccoa_c and hxccoa_x are the same
glutrna_c and glutrna_m are the same
o2_rm and o2_e are the same
udpgal_g and udpg_c are the same
leu__L_m and ile__L_m are the same
hxccoa_rm and hxccoa_x are the same
co2_r and co2_e are the same
inost_e and glc__D_e are the same
dhnpt_m and dhnpt_c are the same
asn__L_c and asn__L_m are the same
amp_n and amp_l are the same
cer2B_26_r and cer2A_26_m are the same
h2o_e and h2o_g are the same
id3acald_c and id3acald_m are the same
amet_rm and amet_n are the same
2phetoh_m and 2phetoh_e are the

HTTPError: 400 Client Error:  for url: https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession,reviewed,id,protein_name,gene_names,organism_name,annotation_score,ec,protein_families,sequence,mass,cc_sequence_caution,ft_chain,cc_ptm,cc_cofactor,cc_subunit,encoded_in,cc_subcellular_location,go_c,ft_intramem,ft_transmem,go_id,keyword,cc_function,cc_activity_regulation,ph_dependence,cc_miscellaneous,cc_alternative_products&format=tsv&query=%28accession:A0A2U9RAK4%20OR%20accession:A0A2U9RAV5%20OR%20accession:A0A2U9RBW3%20OR%20accession:A0A2U9RBK5%20OR%20accession:A0A2U9RCF9%20OR%20accession:A0A2U9RAR9%20OR%20accession:A0A2U9RBJ3%20OR%20accession:A0A2U9RCC6%20OR%20accession:A0A2U9RB22%20OR%20accession:A0A2U9RCH4%20OR%20accession:A0A2U9RAU9%20OR%20accession:A0A2U9RAN3%20OR%20accession:A0A2U9RB35%20OR%20accession:A0A2U9RAL6%20OR%20accession:A0A099NYH5%20OR%20accession:A0A099P8Y8%20OR%20accession:A0A099P8Z3%20OR%20accession:A0A099NV60%20OR%20accession:A0A099P492%20OR%20access

HTTPError: 400 Client Error:  for url: https://rest.uniprot.org/uniprotkb/stream?compressed=true&fields=accession,reviewed,id,protein_name,gene_names,organism_name,annotation_score,ec,protein_families,sequence,mass,cc_sequence_caution,ft_chain,cc_ptm,cc_cofactor,cc_subunit,encoded_in,cc_subcellular_location,go_c,ft_intramem,ft_transmem,go_id,keyword,cc_function,cc_activity_regulation,ph_dependence,cc_miscellaneous,cc_alternative_products&format=tsv&query=%28accession:A0A2U9RAK4%20OR%20accession:A0A2U9RAV5%20OR%20accession:A0A2U9RBW3%20OR%20accession:A0A2U9RBK5%20OR%20accession:A0A2U9RCF9%20OR%20accession:A0A2U9RAR9%20OR%20accession:A0A2U9RBJ3%20OR%20accession:A0A2U9RCC6%20OR%20accession:A0A2U9RB22%20OR%20accession:A0A2U9RCH4%20OR%20accession:A0A2U9RAU9%20OR%20accession:A0A2U9RAN3%20OR%20accession:A0A2U9RB35%20OR%20accession:A0A2U9RAL6%20OR%20accession:A0A099NYH5%20OR%20accession:A0A099P8Y8%20OR%20accession:A0A099P8Z3%20OR%20accession:A0A099NV60%20OR%20accession:A0A099P492%20OR%20access

## Rebuild model, update kapps

In [4]:
# Store the original stoichiometry for comparison later
df_eqn_old = pd.read_csv(os.path.join(wd, 'model', 'RBA_stoichiometry.tsv'), sep='\t', index_col=0)
# df_eqn_old = pd.read_csv('../GAMS/model copy/RBA_stoichiometry.tsv', sep='\t', index_col=0)

In [5]:
%run ./02_build_model.ipynb

Reactions to ignore when checking if enzymes must be added:
['PUNP1_c', 'PUNP1_m', 'PUNP7_c', 'ACONTa_c', 'ASPt2_m', 'TYRtps_v', 'GNNUC_c', 'ILEtps_v', 'HCO3t_c_n', 'CHLSTI_c', 'LEUtps_v', 'GLNtps_v', 'ASPtps_v', 'ASNtps_v', 'GLUtps_v', 'ADNUC_c', 'BIOMASS_MFA_NO_GAM', 'Biomass_MFA_succ_macros_only', 'Biomass_MFA_succ_raw', 'Biomass_MFA_succ_raw_normalized', 'Biomass_MFA_succ_with_MFA_extras', 'Biomass_MFA_succ_with_MFA_extras_with_GAM', 'Biomass_Isor', 'Biomass_MFA', 'BIOMASS_MFA', 'BIOMASS_RBA']
FACOAL240_en
Updating lumpFACS180_c with {'enz': 'JL09_g3954JL09_g50'}
Updating ARABDt_c with {'gene_reaction_rule': ''}
Updating EX_arab__D_e with {'gene_reaction_rule': ''}
Updating EX_csn_e with {'gene_reaction_rule': ''}
Updating CYTK1_c with {'enz': 'JL09_g2829'}
Updating EX_cytd_e with {'gene_reaction_rule': ''}
Updating COAt_c_x with {'gene_reaction_rule': ''}
Updating EX_fru_e with {'gene_reaction_rule': ''}
Updating E4Pt_c_m with {'gene_reaction_rule': ''}
Updating EX_nmn_e with {'ge

/Applications/anaconda3/envs/rba/lib/python3.9/site-packages/cobra/core/metabolite.py:191: UserWarning: The element 'Tr' does not appear in the periodic table
  warn(f"The element {e} does not appear in the periodic table")
/Applications/anaconda3/envs/rba/lib/python3.9/site-packages/cobra/core/metabolite.py:191: UserWarning: The element 'Ma' does not appear in the periodic table
  warn(f"The element {e} does not appear in the periodic table")
/Applications/anaconda3/envs/rba/lib/python3.9/site-packages/cobra/core/metabolite.py:191: UserWarning: The element 'Fc' does not appear in the periodic table
  warn(f"The element {e} does not appear in the periodic table")
/Applications/anaconda3/envs/rba/lib/python3.9/site-packages/cobra/core/metabolite.py:191: UserWarning: The element 'Ap' does not appear in the periodic table
  warn(f"The element {e} does not appear in the periodic table")
/Applications/anaconda3/envs/rba/lib/python3.9/site-packages/cobra/core/metabolite.py:191: UserWarning: 

Enzyme JL09_g3871_c has cofactors {'o2_c'} in its reactions.
  - DAPO_c has cofactor o2_c
  - MAOX_c has cofactor o2_c
  - PAO_c has cofactor o2_c
Enzyme JL09_g1388 has cofactors {'o2_c'} in its reactions.
  - PEAMNO_c has cofactor o2_c
  - PAO_c has cofactor o2_c
  - 4MEABUTNOX_c has cofactor o2_c
  - MAOX_c has cofactor o2_c
  - AACTOOR_c has cofactor o2_c
  - TYMR_c has cofactor o2_c
  - DAPO_c has cofactor o2_c
  - NMPTRCOX_c has cofactor o2_c


In [6]:
df_eqn_new = pd.read_csv(os.path.join(wd, 'model', 'RBA_stoichiometry.tsv'), sep='\t', index_col=0)
# find rxns where 'reaction' column differs between df_eqn_old and df_eqn_new (or rxns exclusive to df_eqn_new) and put them in a list for GAMS to read
# common_idx = df_eqn_new.index.intersection(df_eqn_old.index)
# new_idx = df_eqn_new.index.difference(df_eqn_old.index)
# # find rxns where 'reaction' column isn't present in df_eqn_old
# new_rxns_idx = df_eqn_new.index.difference(df_eqn_old.index)

# changed_idx = common_idx[
# 	df_eqn_new.loc[common_idx, 'reaction'].to_numpy()
# 	!= df_eqn_old.loc[common_idx, 'reaction'].to_numpy()
# ]

# df_eqn_diff = df_eqn_new.loc[
# 	# df_eqn_new.index.isin(new_idx.union(changed_idx))
# 	df_eqn_new.index.isin(changed_idx)
# ]
df_eqn_diff_list = df_eqn_new[df_eqn_new['reaction'].isin(df_eqn_old['reaction'].values) == False].index.tolist()
# df_eqn_diff_list = df_eqn_diff.index.tolist()
with open('../GAMS/model/decompartmentalized_rxns.txt', 'w') as f:
	f.write('/\n')
	for rxn in df_eqn_diff_list:
		f.write(f"'{rxn}'\n")
	f.write('/')

In [7]:
#TODO: Ensure MPFS constraints are updated to work for decompartmentalized model too
%run ../parameterization/kapp/datasets/ShenEtAl2024_batch/B2_min_flux_violation.py
%run ../parameterization/kapp/datasets/ShenEtAl2024_batch/C1_calculate_kapp.py

Using mu = 0.4773921790158572 1/hr based on flux data file /Users/ejm6426/Documents/pkRBA/parameterization/kapp/datasets/raw_data_files/fluxes_net.tsv


FileNotFoundError: [Errno 2] No such file or directory: './GAMS_setting_files/min_flux_violation_GAMS_settings.txt'

Is 2-column format used? True
BIOSYN-BIODILAERO-NOGAM is not found in raw flux. No growth rate assigned
Growth rate: 0.47739217901585723
Median kapp (1/h): 25237.767562429875
Using default kapp for ENZLOAD-3HACD200_rm_FWD-JL09_g460_rm since no kapp could be calculated.
Using default kapp for ENZLOAD-3HACD220_rm_FWD-JL09_g460_rm since no kapp could be calculated.
Using default kapp for ENZLOAD-3HACD240_rm_FWD-JL09_g460_rm since no kapp could be calculated.
Using default kapp for ENZLOAD-3HACD260_rm_FWD-JL09_g460_rm since no kapp could be calculated.
Using default kapp for ENZLOAD-3OACE200_rm_FWD-JL09_g3004_rm since no kapp could be calculated.
Using default kapp for ENZLOAD-3OACE220_rm_FWD-JL09_g3004_rm since no kapp could be calculated.
Using default kapp for ENZLOAD-3OACE240_rm_FWD-JL09_g3004_rm since no kapp could be calculated.
Using default kapp for ENZLOAD-3OACE260_rm_FWD-JL09_g3004_rm since no kapp could be calculated.
Using default kapp for ENZLOAD-ACt_c_m_REV-JL09_g191 since no

sh: module: command not found
sh: gams: command not found
